# Ridge Regression - Key Concepts

## 1. How the Coefficients Get Affected

### Ridge Regression Cost Function

$$
J(\beta_0,\beta_1,\dots,\beta_p)
=
\sum_{i=1}^{n}
\left(
y_i -
\left(
\beta_0 + \beta_1x_{i1} + \beta_2x_{i2} + \dots + \beta_px_{ip}
\right)
\right)^2
+
\lambda
\sum_{j=1}^{p}\beta_j^2
$$

Where:

- \( y_i \) = Actual value  
- \( \hat{y}_i \) = Predicted value  
- \( \beta_j \) = Coefficients  
- \( \lambda \) = Regularization parameter  

### Effect
- Coefficients shrink toward zero.
- Prevents overfitting.

---

# 2. Higher Values are Impacted More

### Penalty Term

$$
\lambda
\left(
\beta_1^2 + \beta_2^2 + \dots + \beta_p^2
\right)
$$

If coefficients become large:

$$
\beta_j^2 \uparrow
\quad \Rightarrow \quad
\text{Penalty} \uparrow
$$

### Effect
- Large coefficients receive stronger punishment.
- Model becomes more stable.

---

# 3. Bias-Variance Tradeoff

### Total Error Formula

$$
\text{Total Error}
=
\text{Bias}^2
+
\text{Variance}
+
\text{Irreducible Error}
$$

### With Ridge Regression

If:

$$
\lambda \uparrow
$$

Then:

$$
\text{Bias} \uparrow
$$

and

$$
\text{Variance} \downarrow
$$

### Meaning
- Small \( \lambda \) → flexible model
- Large \( \lambda \) → simpler model

---

# 4. Contour Plots

### Ridge Constraint Region

$$
\sum_{j=1}^{p}\beta_j^2 \leq t
$$

For 2 variables:

$$
\beta_1^2 + \beta_2^2 \leq t
$$

This forms a circular region.

### Interpretation
- Ridge shrinks coefficients smoothly.
- Most coefficients remain non-zero.

---

## 5. Why it is Called Ridge Regression
- The penalty term creates a "ridge" in the cost function surface.
- This stabilizes coefficient estimation.
- Especially useful when multicollinearity exists.

### Multicollinearity
- Happens when independent variables are highly correlated.
- Ridge reduces instability caused by correlated features.

---

# Practical Tip

## Standardization Formula

$$
z
=
\frac{x-\mu}{\sigma}
$$

Where:

- \( \mu \) = Mean  
- \( \sigma \) = Standard deviation  


- Always standardize features before applying Ridge Regression.
- Use cross-validation to choose the best \(\lambda\) value.
- Works well when:
  - Many features exist
  - Features are correlated
  - Overfitting is present

# 1. How coefficients are affected?


In [ ]:
from sklearn.datasets import load_diabetes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = load_diabetes()

In [ ]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['TARGET'] = data.target

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=2)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

In [ ]:
coefs = []
r2_scores = []

for i in [0, 10, 100, 1000]:
    reg = Ridge(alpha=i)
    reg.fit(X_train, y_train)
    
    coefs.append(reg.coef_.tolist())
    y_pred = reg.predict(X_test)
    r2_scores.append(r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(14, 9))

plt.subplot(221)
plt.bar(data.feature_names, coefs[0])
plt.title('Alpha = 0, R² score = {}'.format(round(r2_scores[0], 2)))
plt.xticks(rotation=45)

plt.subplot(222)
plt.bar(data.feature_names, coefs[1])
plt.title('Alpha = 10, R² score = {}'.format(round(r2_scores[1], 2)))
plt.xticks(rotation=45)

plt.subplot(223)
plt.bar(data.feature_names, coefs[2])
plt.title('Alpha = 100, R² score = {}'.format(round(r2_scores[2], 2)))
plt.xticks(rotation=45)

plt.subplot(224)
plt.bar(data.feature_names, coefs[3])
plt.title('Alpha = 1000, R² score = {}'.format(round(r2_scores[3], 2)))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 2. Higher Coefficients are affected more

In [ ]:
alphas = [0, 0.0001, 0.0005, 0.001, 0.005, 0.1, 0.5, 1, 5, 10]
coefs = []

for i in alphas:
    reg = Ridge(alpha=i)
    reg.fit(X_train, y_train)
    coefs.append(reg.coef_.tolist())

In [ ]:
input_array = np.array(coefs).T  # Transpose to get (n_features, n_alphas)

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(alphas, np.zeros(len(alphas)), color='black', linewidth=3, label='Zero line')

for i in range(input_array.shape[0]):
    plt.plot(alphas, input_array[i], marker='o', label=data.feature_names[i], linewidth=2)

plt.xlabel('Alpha (Regularization Parameter)', fontsize=12)
plt.ylabel('Coefficient Value', fontsize=12)
plt.title('Effect of Alpha on Ridge Coefficients', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

# 3. Impact on Bias and Variance

In [ ]:
# Generate synthetic data
m = 100
X = 5 * np.random.rand(m, 1) - 2
y = 0.7 * X ** 2 - 2 * X + 3 + np.random.randn(m, 1)

plt.figure(figsize=(8, 6))
plt.scatter(X, y, alpha=0.6)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Sample Data for Bias-Variance Analysis')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X.reshape(100, 1), y.reshape(100), test_size=0.2, random_state=2
)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Create polynomial features
poly = PolynomialFeatures(degree=15)
X_train_poly_transformed = poly.fit_transform(X_train_poly)
X_test_poly_transformed = poly.transform(X_test_poly)

print(f"Original X_train shape: {X_train_poly.shape}")
print(f"Polynomial X_train shape: {X_train_poly_transformed.shape}")

In [ ]:
# Install mlxtend if not already installed
# pip install mlxtend

try:
    from mlxtend.evaluate import bias_variance_decomp
    
    alphas = np.linspace(0, 30, 50)
    loss = []
    bias = []
    variance = []
    
    for i in alphas:
        reg = Ridge(alpha=i)
        avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
            reg, X_train_poly_transformed, y_train_poly, 
            X_test_poly_transformed, y_test_poly, 
            loss='mse', random_seed=123, num_rounds=10
        )
        loss.append(avg_expected_loss)
        bias.append(avg_bias)
        variance.append(avg_var)
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(alphas, loss, label='Total Loss', linewidth=2, marker='o')
    plt.plot(alphas, bias, label='Bias²', linewidth=2, marker='s')
    plt.plot(alphas, variance, label='Variance', linewidth=2, marker='^')
    plt.ylim(0, max(loss) * 1.1)
    plt.xlabel('Alpha (Regularization Parameter)', fontsize=12)
    plt.ylabel('Error', fontsize=12)
    plt.title('Bias-Variance Tradeoff in Ridge Regression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()
    
except ImportError:
    print("mlxtend not installed. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'mlxtend'])
    print("Please re-run this cell after installation.")

# 4. Effect of Regularization on Loss Function

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# Generate regression data
X_reg, y_reg = make_regression(
    n_samples=100, n_features=1, n_informative=1, n_targets=1, 
    noise=20, random_state=13
)

plt.figure(figsize=(10, 6))
plt.scatter(X_reg, y_reg, alpha=0.6, s=50)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Regression Data')
plt.grid(True, alpha=0.3)
plt.show()

# Fit linear regression
reg = LinearRegression()
reg.fit(X_reg, y_reg)
print(f"Coefficient: {reg.coef_[0]:.4f}")
print(f"Intercept: {reg.intercept_:.4f}")

In [ ]:
# Define loss function with regularization
def cal_loss(m, alpha):
    """Calculate loss: sum((y - y_pred)^2) + alpha * m^2"""
    y_pred = m * X_reg.ravel() + reg.intercept_
    mse_loss = np.sum((y_reg - y_pred) ** 2)
    regularization = alpha * (m ** 2)
    return mse_loss + regularization

In [ ]:
# Create loss curves for different alpha values
m_values = np.linspace(-100, 150, 200)
alphas_loss = [0, 10, 20, 30, 40, 50, 100]

plt.figure(figsize=(12, 7))
for alpha_val in alphas_loss:
    loss_values = [cal_loss(m, alpha_val) for m in m_values]
    plt.plot(m_values, loss_values, label=f'α = {alpha_val}', linewidth=2, marker='', alpha=0.8)

plt.xlabel('Coefficient (m)', fontsize=12)
plt.ylabel('Total Loss (MSE + Regularization)', fontsize=12)
plt.title('Effect of Regularization Parameter on Loss Function', fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- As alpha increases, the ridge (minimum point) moves closer to 0")
print("- The loss function becomes wider/flatter")
print("- This demonstrates the shrinkage effect of ridge regression")